# Introduction

Training a model across several GPUs or computers means sharing the work. Each worker
handles part of the computation, but it will often need something another worker has
computed. Getting those pieces to work together is what this course is about.

Imagine each worker has calculated a partial sum. To get the total, someone needs to
collect and combine those values. If every worker needs the total for its next step,
the answer has to reach all of them. Even this small task raises questions about who
sends, who receives, and when each worker can continue.

In these puzzles, you'll build the communication patterns behind distributed training.
We'll begin with ordinary Python values, then use the same ideas to combine tensor
results and divide larger computations among workers. The aim is to be able to look at
a distributed program and reason about what each worker knows, what it can do locally,
and what it is waiting for.

## A place to experiment

You can write real distributed programs with libraries such as `torch.distributed` and
MPI. Alongside the communication algorithm, you'll need to arrange how processes start
and connect, and where they run. When a program hangs, you may have to untangle setup
problems and communication mistakes at the same time. That's a lot to keep in your head
while you're still learning how an exchange works.

We'll use **Flock** to make those first experiments easier. It runs all workers in one
Python process, so you can write and inspect their interactions in a notebook without
setting up a cluster. Its API borrows ideas from `torch.distributed` and MPI, with a
smaller execution model for teaching. You still decide what to send, what to receive,
and how to let every worker finish.

That convenience comes with a limit: Flock doesn't spread your Python computation across
machines or simulate network performance. It won't show the real speedup of adding CPUs
or GPUs. What you can explore here is whether the workers exchange the right data and
whether their communication lets the program make progress.

## Running several workers

To write one of these programs, we need a way for each worker to identify itself.
Flock gives every worker a number called its **rank**. With three workers, the ranks are
`0`, `1`, and `2`. `flock.get_rank()` returns that number, and `flock.get_world_size()`
returns the total number of workers.

We write one function, and every rank runs its own call to it. You'll notice an extra
keyword in its definition: `async`.

> **A little scheduling magic: `async` and `await`**
>
> If rank 0 is waiting for rank 1, we need to pause rank 0 while rank 1 gets a chance
> to run. `async def` makes a function capable of pausing and resuming; `await` gives
> it a place to wait for an operation. When communication can't complete yet, Flock
> can run another rank and come back to this one later, with its local variables intact.
>
> You can leave that machinery as a little magic for now. For the puzzles, read
> `await` as “this rank needs the operation to finish before continuing.” Flock handles
> the scheduling. For an optional deep dive, Victor Skvortsov's
> [How async/await works in Python](https://tenthousandmeters.com/blog/python-behind-the-scenes-12-how-asyncawait-works-in-python/)
> builds up to `async`/`await` through event loops, generators, and `yield from`.

Our first function has nothing to wait for yet. Let's see what comes back when three
ranks each return a greeting:

In [ ]:
import flock


async def hello_world(msg: str) -> str:
    return f"I'm rank {flock.get_rank()} of {flock.get_world_size()}, {msg}"


run = flock.distribute(workers=3)(hello_world)
run("hello world!")


`flock.distribute(workers=3)(hello_world)` creates a function that runs `hello_world` on
three ranks. Calling it passes the same argument to each rank and collects their results
in rank order, regardless of which one finishes first:

```text
["I'm rank 0 of 3, hello world!", "I'm rank 1 of 3, hello world!", "I'm rank 2 of 3, hello world!"]
```

For workers to handle different parts of a task, they'll need different inputs.
`flock.per_rank(...)` supplies one argument per rank:

In [ ]:
run(flock.per_rank(["hello from rank 0!", "hello from rank 1!", "hello from rank 2!"]))

The first message goes to rank 0, the second to rank 1, and the third to rank 2.
Each call has its own arguments and local variables. This way of running the same
program with different data is called **SPMD: single program, multiple data**.

Running the same function doesn't mean ranks move through it together. They can take
different branches, and one can be waiting for a message while another continues.
The result list above is collected by `run(...)` outside the workers; a worker doesn't
automatically get to see the other workers' results.

## Sending and receiving

Suppose we want two ranks to exchange greetings, so each returns the greeting written
by the other. Returning their own strings to the caller won't accomplish that: the
values need to travel between the ranks.

- `await flock.send(peer, value)` sends a value to another rank.
- `await flock.recv(peer)` receives a value from that rank and returns it.

In Flock, both operations are **blocking**: a send waits for a matching receive, and a
receive waits for a matching message. Waiting pauses the rank making the call. Flock
can then run another rank that has work to do.

Here, rank 0 sends its greeting first. Rank 1 receives it, then sends a greeting back:

In [ ]:
async def exchange() -> str:
    if flock.get_rank() == 0:
        await flock.send(1, "hello from rank 0!")
        message = await flock.recv(1)
        return f'rank {flock.get_rank()} got the message "{message}"'

    if flock.get_rank() == 1:
        message = await flock.recv(0)
        await flock.send(0, "hello from rank 1!")
        return f'rank {flock.get_rank()} got the message "{message}"'

    raise RuntimeError("Unreachable")

In [ ]:
run = flock.distribute(workers=2)(exchange)
run()

The first exchange pairs rank 0's send with rank 1's receive. The second pairs rank 1's
send with rank 0's receive. Both ranks reach the operations their peer is waiting for.

If both ranks started with `await flock.send(...)`, each would wait for the other to
receive. Neither would reach its next line. This is a **deadlock**.

## Starting an operation without waiting

The blocking exchange works because the two ranks do things in a different order.
To let both ranks send first, we need to separate starting a send from waiting for it
to complete.

`flock.isend(...)` starts a send and immediately returns a `Work` object.
`flock.irecv(...)` does the same for a receive. Neither call needs `await`.
Later, `await work.wait()` waits for completion; for a receive, it also returns the message.

Both ranks can now use the same code: start a send, start a receive, then wait for both.
With two ranks, `1 - rank` gives the other rank's ID:

In [ ]:
async def async_exchange() -> str:
    rank = flock.get_rank()
    peer = 1 - rank
    send_work = flock.isend(peer, f"hello from rank {rank}!")
    recv_work = flock.irecv(peer)

    message = await recv_work.wait()
    await send_work.wait()
    return f'rank {rank} got the message "{message}"'


In [ ]:
run = flock.distribute(workers=2)(async_exchange)
run()

Each rank starts both operations before waiting, so the other rank can reach a matching
receive. Keep each `Work` object and await its `wait()` before returning, even if the
communication has already matched.

Using `isend` alone doesn't prevent deadlocks. If both ranks immediately waited on their
sends before starting their receives, they would get stuck again. The useful change is
that each rank can reach its receive while its send is still pending.

## Exchanging tensors

In a training computation, the value another worker needs will often be a tensor.
We can keep the exchange above and change what it carries.

`flock` communicates ordinary Python values. For tensor communication, use `flock.torch`,
imported here as `ft`. We still use `flock` to launch workers and identify ranks, and
`torch` to create tensors:

In [ ]:
import flock
import flock.torch as ft
import torch

In [ ]:
async def tensor_exchange(x: torch.Tensor) -> torch.Tensor:
    peer = 1 - flock.get_rank()
    send_work = ft.isend(peer, x)
    recv_work = ft.irecv(peer)
    received = await recv_work.wait()
    await send_work.wait()
    return received


run = flock.distribute(workers=2)(tensor_exchange)
run(flock.per_rank([torch.tensor([10, 20]), torch.tensor([30, 40])]))


The results are returned in rank order:

```text
[tensor([30, 40]), tensor([10, 20])]
```

Each rank has received a value it didn't start with. In [Messages between ranks](01_p2p.ipynb),
you'll work out the exchanges yourself as more workers take part. The question to keep
asking is what each worker needs before it can take its next step.